In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from dotenv import load_dotenv

import janitor

import os

from pathlib import Path

import openpyxl

import sqlalchemy as sa


# Desactivar notación científica

pd.set_option('display.float_format', lambda x: '%.3f' % x)

np.set_printoptions(suppress=True)


# Cargar variables de entorno

load_dotenv()


print("✅ Librerías importadas correctamente")

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve().parent
raw_dir = project_root / '02_datos' / '01_Originales'
files = [
    raw_dir / 'salesdaily.csv',
    raw_dir / 'saleshourly.csv',
    raw_dir / 'salesmonthly.csv',
    raw_dir / 'salesweekly.csv',
]

for file in files:
    print(f'\n=== {file.name} ===')
    df = pd.read_csv(file)
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print(df.head().to_string(index=False))
    print('dtypes:')
    print(df.dtypes)
    print('---')



=== salesdaily.csv ===
shape: (2106, 13)
columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']
   datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C  R03  R06  Year  Month  Hour Weekday Name
1/2/2014    0.0   3.67    3.4  32.40   7.0   0.0  0.0  2.0  2014      1   248     Thursday
1/3/2014    8.0   4.00    4.4  50.60  16.0   0.0 20.0  4.0  2014      1   276       Friday
1/4/2014    2.0   1.00    6.5  61.85  10.0   0.0  9.0  1.0  2014      1   276     Saturday
1/5/2014    4.0   3.00    7.0  41.10   8.0   0.0  3.0  0.0  2014      1   276       Sunday
1/6/2014    5.0   1.00    4.5  21.70  16.0   2.0  6.0  2.0  2014      1   276       Monday
dtypes:
datum            object
M01AB           float64
M01AE           float64
N02BA           float64
N02BE           float64
N05B            float64
N05C            float64
R03             float64
R06             float64
Year              int64
Month             int64
Hour          

In [2]:
from pathlib import Path
import pandas as pd

orig_dir = Path.cwd().resolve().parent / '02_datos' / '01_Originales'
files_info = [
    ('saleshourly.csv','hour'),
    ('salesdaily.csv','day'),
    ('salesweekly.csv','week'),
    ('salesmonthly.csv','month'),
]

dfs = []
for fname, gran in files_info:
    fp = orig_dir / fname
    print('Loading', fp.name)
    tmp = pd.read_csv(fp)
    tmp['granularity'] = gran
    # detect a date-like column
    date_cols = [c for c in tmp.columns if c.lower() in ('date','fecha','ds','timestamp','time')]
    if date_cols:
        tmp['date'] = pd.to_datetime(tmp[date_cols[0]], errors='coerce')
    else:
        tmp['date'] = pd.to_datetime(tmp.iloc[:,0], errors='coerce')
    dfs.append(tmp)

# Concatenate vertical
df = pd.concat(dfs, ignore_index=True, sort=False)

print('\nCombined dataframe')
print('shape:', df.shape)
print('columns:', list(df.columns))
print(df.head().to_string(index=False))
print('\ndtypes:')
print(df.dtypes)

# Save combined to caches
cache_path = Path.cwd().resolve().parent / '02_datos' / '04_Caches' / '01_combined_tablon.pkl'
cache_path.parent.mkdir(parents=True, exist_ok=True)
df.to_pickle(cache_path)
print('\nSaved combined dataframe to', cache_path)


Loading saleshourly.csv
Loading salesdaily.csv
Loading salesweekly.csv
Loading salesmonthly.csv

Combined dataframe
shape: (53010, 15)
columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name', 'granularity', 'date']
         datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C  R03  R06   Year  Month  Hour Weekday Name granularity                date
 1/2/2014 8:00    0.0   0.67    0.4    2.0   0.0   0.0  0.0  1.0 2014.0    1.0   8.0     Thursday        hour 2014-01-02 08:00:00
 1/2/2014 9:00    0.0   0.00    1.0    0.0   2.0   0.0  0.0  0.0 2014.0    1.0   9.0     Thursday        hour 2014-01-02 09:00:00
1/2/2014 10:00    0.0   0.00    0.0    3.0   2.0   0.0  0.0  0.0 2014.0    1.0  10.0     Thursday        hour 2014-01-02 10:00:00
1/2/2014 11:00    0.0   0.00    0.0    2.0   1.0   0.0  0.0  0.0 2014.0    1.0  11.0     Thursday        hour 2014-01-02 11:00:00
1/2/2014 12:00    0.0   2.00    0.0    5.0   2.0   0.0  0.0  0.0 2

In [4]:
from pathlib import Path
import pandas as pd
from dateutil.relativedelta import relativedelta

# Uso de split temporal por fecha (time-based split) como clave primaria lógica para forecasting
# Carga combinado desde caches
project_root = Path.cwd().resolve().parent
cache_path = project_root / '02_datos' / '04_Caches' / '01_combined_tablon.pkl'
df = pd.read_pickle(cache_path)

# Asegurar datetime y ordenar
if 'date' not in df.columns:
    raise ValueError('La columna `date` no existe en el dataframe combinado')
df = df.sort_values('date').reset_index(drop=True)

# Split temporal: últimos 3 meses en validación
max_date = df['date'].max()
cutoff_date = max_date - relativedelta(months=3)
train = df[df['date'] < cutoff_date].copy()
validation = df[df['date'] >= cutoff_date].copy()

# Rutas de salida
train_path = project_root / '02_datos' / '03_Entrenamiento' / '01_train_tablon_integrado.pkl'
val_path = project_root / '02_datos' / '02_Validacion' / 'validation.pkl'
train_path.parent.mkdir(parents=True, exist_ok=True)
val_path.parent.mkdir(parents=True, exist_ok=True)
train.to_pickle(train_path)
validation.to_pickle(val_path)

print('Split realizado (time-based: últimos 3 meses)')
print('Total rows:', len(df))
print('Fecha mínima:', df['date'].min())
print('Fecha máxima:', df['date'].max())
print('Fecha de corte:', cutoff_date)
print('Train shape:', train.shape)
print('Validation shape:', validation.shape)
print('\nGuardados:')
print('-', train_path)
print('-', val_path)


Split realizado (time-based: últimos 3 meses)
Total rows: 53010
Fecha mínima: 2014-01-02 00:00:00
Fecha máxima: 2019-10-31 00:00:00
Fecha de corte: 2019-07-31 00:00:00
Train shape: (51249, 15)
Validation shape: (1761, 15)

Guardados:
- C:\Users\sergi\Desktop\DATA\CURSO DS4B\EstructuraDirectorio\Pharma Sales\02_datos\03_Entrenamiento\01_train_tablon_integrado.pkl
- C:\Users\sergi\Desktop\DATA\CURSO DS4B\EstructuraDirectorio\Pharma Sales\02_datos\02_Validacion\validation.pkl


In [5]:
import io
import sys
from pathlib import Path

# Cargar el dataframe combinado
project_root = Path.cwd().resolve().parent
cache_path = project_root / '02_datos' / '04_Caches' / '01_combined_tablon.pkl'
df = pd.read_pickle(cache_path)

# Capturar df.info() en string
buffer = io.StringIO()
df.info(buf=buffer)
info_output = buffer.getvalue()

print("DataFrame Info:\n")
print(info_output)

# Actualizar copilot-instructions.md
instructions_path = project_root / '.github' / 'copilot-instructions.md'
with open(instructions_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Buscar si ya existe sección de df.info
if '## DataFrame Schema' not in content:
    # Añadir sección al final
    new_section = f'\n## DataFrame Schema\n\n```\n{info_output}\n```\n'
    content += new_section
    with open(instructions_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"✅ Actualizado: {instructions_path}")
else:
    print("⚠️ La sección 'DataFrame Schema' ya existe en copilot-instructions.md")


DataFrame Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53010 entries, 0 to 53009
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   datum         53010 non-null  object        
 1   M01AB         53010 non-null  float64       
 2   M01AE         53010 non-null  float64       
 3   N02BA         53010 non-null  float64       
 4   N02BE         53010 non-null  float64       
 5   N05B          53010 non-null  float64       
 6   N05C          53010 non-null  float64       
 7   R03           53010 non-null  float64       
 8   R06           53010 non-null  float64       
 9   Year          52638 non-null  float64       
 10  Month         52638 non-null  float64       
 11  Hour          52638 non-null  float64       
 12  Weekday Name  52638 non-null  object        
 13  granularity   53010 non-null  object        
 14  date          53010 non-null  datetime64[ns]
dtypes: datetime64[ns](1